# Notebook 02: Expert Activation Skew & Load Imbalance Analysis
This notebook visualizes which MoE experts get routed most frequently by the gating network, calculating expert imbalance metrics.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

### Load Profiler Outputs

In [ ]:
report_path = "../results/profiling/expert_imbalance.json"
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
else:
    # Simulated zipf-like report
    num_experts = 64
    ranks = np.arange(1, num_experts + 1)
    counts = 100000 / (ranks ** 0.75)
    counts = np.round(counts).astype(int)
    np.random.shuffle(counts)
    report = {
        "num_experts": num_experts,
        "imbalance_ratio": max(counts) / np.mean(counts),
        "activation_counts": {str(i): int(counts[i]) for i in range(num_experts)}
    }

print(f"Imbalance Ratio: {report['imbalance_ratio']:.3f}")

### Plot Expert Activation Frequencies

In [ ]:
counts_dict = report["activation_counts"]
expert_ids = list(map(int, counts_dict.keys()))
frequencies = list(counts_dict.values())

sorted_idx = np.argsort(frequencies)[::-1]
sorted_ids = [expert_ids[i] for i in sorted_idx]
sorted_freqs = [frequencies[i] for i in sorted_idx]

plt.figure(figsize=(12, 5))
plt.bar(range(len(sorted_freqs)), sorted_freqs, color='darkcyan', width=0.8)
plt.title("Expert Activation Distribution (Sorted by Count)", fontsize=13, fontweight='bold')
plt.xlabel("Expert Rank")
plt.ylabel("Routing Activation Counts")
plt.tight_layout()
plt.show()